# 1.5B **policy** — pushed-further PPO (v2) vs the 0.8025 reward model

v1 (lr 1e-6, 1024 ep, tight leash) was **judge-validated at 59.25%** but KL/seq stayed ~0.03 — far 
under the target-3 budget, so the policy barely used its leash. **v2 pushes harder: lr 1e-6→5e-6, 
1024→2048 episodes**, *same* anti-hack guards (KL target 3, length/EOS penalties, score_clip). 
Question: does using more of the KL budget buy more *real* (judge-validated) win — or tip into Goodhart?

PPO-only, **~3 h on a T4**. On-Kaggle win-rate is RM-judged (circular) — judge-validate locally after.

**Prerequisite:** `+ Add Input` the dataset `georgezhang06/rlhf-rm-1p5b-08025` (verified 0.8025, 3 GB).

In [ ]:
import torch, json
cap=torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0,0)
name=torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'
P100=tuple(cap)==(6,0); json.dump({'p100':P100},open('/tmp/gpu.json','w'))
print(f'GPU: {name} cap={cap} -> '+('P100 fp32' if P100 else 'T4 bf16'))

In [ ]:
!pip install -q 'transformers>=4.44,<5' 'datasets>=2.20' 'accelerate>=0.33' 'peft>=0.12' pytest
import json, subprocess, sys
if json.load(open('/tmp/gpu.json'))['p100']:
    subprocess.run([sys.executable,'-m','pip','install','-q','transformers>=4.44,<4.48'],check=True)
    subprocess.run([sys.executable,'-m','pip','install','-q','torch==2.4.1','torchvision==0.19.1',
                    'torchaudio==2.4.1','--index-url','https://download.pytorch.org/whl/cu121'],check=True)

In [ ]:
import os
!rm -rf /kaggle/working/rlhf-pipeline && git clone -q https://github.com/TheYellowDuck/RLHF-pipeline.git /kaggle/working/rlhf-pipeline
%cd /kaggle/working/rlhf-pipeline

In [ ]:
!python scripts/smoke_test.py 2>&1 | tail -2

## Config + locate the attached 0.8025 reward model

In [ ]:
import glob, os, shutil, json
P100=json.load(open('/tmp/gpu.json'))['p100']
MODEL='Qwen/Qwen2.5-1.5B-Instruct'
DATA_CLEAN='argilla/ultrafeedback-binarized-preferences-cleaned'
DTYPE,BF16=('float32','false') if P100 else ('bfloat16','true')
hits=glob.glob('/kaggle/input/**/reward_config.json', recursive=True)
assert hits, 'No reward model under /kaggle/input. + Add Input -> georgezhang06/rlhf-rm-1p5b-08025.'
def _wsz(d):
    w=glob.glob(d+'/*.safetensors')+glob.glob(d+'/*.bin')
    return max([os.path.getsize(f) for f in w], default=0)
cands=sorted((os.path.dirname(h) for h in hits if '/smoke/' not in h), key=_wsz, reverse=True)
assert cands, 'only smoke checkpoints found under /kaggle/input'
RM_SRC=cands[0]
os.makedirs('checkpoints', exist_ok=True)
if os.path.abspath(RM_SRC)!=os.path.abspath('checkpoints/reward_model'):
    shutil.rmtree('checkpoints/reward_model', ignore_errors=True)
    shutil.copytree(RM_SRC, 'checkpoints/reward_model')
sz=_wsz('checkpoints/reward_model')
print(f'RM <- {RM_SRC} ({sz/1e6:.0f} MB weights); policy={MODEL} dtype={DTYPE}')
assert sz>1e8, 'RM weights too small/partial for a 1.5B model — re-create the Dataset from a COMPLETE output.'

## PPO v2 — push harder (lr 5e-6, 2048 episodes), same anti-hack leash (~3 h)

In [ ]:
!python scripts/train_ppo.py \
  -o policy.name_or_path=$MODEL -o policy.dtype=$DTYPE -o policy.use_lora=true \
  -o reward_model.name_or_path=checkpoints/reward_model -o reward_model.dtype=$DTYPE \
  -o data.name=$DATA_CLEAN -o data.train_split='train[2000:]' -o data.max_samples=6000 \
  -o ppo.total_episodes=2048 -o ppo.rollout_batch_size=8 -o ppo.mini_batch_size=1 \
  -o ppo.lr=5e-6 -o ppo.normalize_rewards=true -o ppo.save_every=300 \
  -o ppo.kl.target=3.0 -o ppo.kl.init_coef=0.3 \
  -o ppo.length_penalty=0.01 -o ppo.missing_eos_penalty=1.0 -o ppo.score_clip=8.0 \
  -o generation.max_new_tokens=40 -o data.max_prompt_length=160
# v2: 5x lr + 2x episodes to actually move into the KL budget (v1 used only ~1% of it), same leash.
# Watch ppo/reward/kl_seq -> should rise toward ~1-3 now. OOM? drop -o ppo.rollout_batch_size=4.

## Evaluate — PPO vs base Instruct, RM-judged win-rate -> RESULTS.md

In [ ]:
import subprocess
def run(c):
    print('$',c,flush=True); o=subprocess.run(c,shell=True,capture_output=True,text=True)
    print(o.stdout[-700:])
    if o.returncode: print(o.stderr[-1200:])
    return o.stdout
wn=run(f"python scripts/evaluate.py score-policy --policy checkpoints/ppo "
        f"--reward-model checkpoints/reward_model --compare {MODEL} --data {DATA_CLEAN} "
        f"--split 'train[2000:]' --num 100 --max-new-tokens 40 --max-length 320")
md=(f'# 1.5B PPO v2 results (lr 5e-6, 2048 ep, same leash)\n\n- policy: `{MODEL}` (LoRA, PPO) vs base `{MODEL}`\n'
    f'- reward model: the 0.8025 1.5B RM\n- v1 was judge-validated 59.25%; v2 pushes harder\n\n'
    f'## PPO vs Instruct - RM-judged win-rate + samples (CIRCULAR — validate with the local Claude judge)\n'
    f'```\n{wn}\n```\n')
open('/kaggle/working/RESULTS.md','w').write(md); print('\nSaved -> RESULTS.md')

### Read — then JUDGE-VALIDATE locally (the honest number)
Download `checkpoints/ppo`, then run the independent judge (key in `.env`):
```
./.venv/bin/python scripts/evaluate.py judge --policy <ppo_ckpt> \
  --base Qwen/Qwen2.5-1.5B-Instruct \
  --data argilla/ultrafeedback-binarized-preferences-cleaned --split 'train[2000:]' \
  --num 100 --max-new-tokens 40 --device cpu
```
Judge **> 59.25%** = pushing harder bought *real* extra win; **≈59%** = v1 already near the ceiling for 
this RM; **< 50%** = it tipped into Goodhart (then the RM, not the PPO, is the ceiling).